# 05 — Model Benchmarking and Evaluation

Goal: benchmark required model families, compare metrics, and select serving candidate.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pipeline').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'pipeline').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

Project root: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/MLOps, UI, and Deployment/Deploy a Machine Learning Model with Docker


## 5.1 Benchmark Strategy
**Definition:** Benchmark combines manual baselines with AutoML accelerators for stronger model selection evidence.

**Theory:** Explain mechanism, assumptions, and where this fits in deployment lifecycle.

**Motivation:** Why this matters for reliability, reproducibility, and operations.

**Real-World Example:** Manual models give interpretability; AutoML finds high-performing configs under budget.

**Visual Explanation:** See figure/code cell below.

**Code Explanation:** Code cell demonstrates concrete implementation details.

**Best Practices:** Track multiple metrics not just R².

**Common Mistakes:** Choosing model by single metric without error-profile context.

In [2]:
from pipeline.benchmarking import run_training_pipeline

# Use fast profile for notebook executability; use deep profile for portfolio-grade final runs.
artifacts = run_training_pipeline(profile='fast')
print('Serving model:', artifacts.model_name)
artifacts.ranking.head(10)

2026-06-25 03:08:29,085 | INFO | pipeline.benchmarking | Training manual model: Linear Regression


2026-06-25 03:08:29,090 | INFO | pipeline.benchmarking | Training manual model: Ridge


2026-06-25 03:08:29,094 | INFO | pipeline.benchmarking | Training manual model: Lasso


2026-06-25 03:08:29,119 | INFO | pipeline.benchmarking | Training manual model: Random Forest


2026-06-25 03:08:31,038 | INFO | pipeline.benchmarking | Training manual model: Extra Trees


...

2026-06-25 03:08:34,880 | INFO | pipeline.benchmarking | Training manual model: XGBoost


2026-06-25 03:08:39,436 | INFO | pipeline.benchmarking | Training manual model: LightGBM


2026-06-25 03:08:39,613 | INFO | pipeline.benchmarking | Training manual model: CatBoost


Serving model: LightGBM


,Model,Source,MAE,MSE,RMSE,R²,MAPE
0,LightGBM,Manual,0.308637,0.214585,0.463233,0.836246,0.181080
1,XGBoost,Manual,0.313379,0.221837,0.470996,0.830712,0.181579
2,Extra Trees,Manual,0.325320,0.251297,0.501295,0.808230,0.184118
3,CatBoost,Manual,0.338217,0.252274,0.502269,0.807484,0.194437
4,Random Forest,Manual,0.327651,0.254869,0.504845,0.805505,0.189125
5,Gradient Boosting,Manual,0.371643,0.293997,0.542215,0.775645,0.215254
6,Lasso,Manual,0.533286,0.553894,0.744241,0.577312,0.319538
7,Ridge,Manual,0.533204,0.555803,0.745522,0.575855,0.319523
8,Linear Regression,Manual,0.533200,0.555892,0.745581,0.575788,0.319522


## 5.2 Metric Interpretation (MAE/MSE/RMSE/R²/MAPE)
**Definition:** Different metrics emphasize different error behaviors and business impacts.

**Theory:** Explain mechanism, assumptions, and where this fits in deployment lifecycle.

**Motivation:** Why this matters for reliability, reproducibility, and operations.

**Real-World Example:** MAPE useful for relative error interpretation but unstable near zero targets.

**Visual Explanation:** See figure/code cell below.

**Code Explanation:** Code cell demonstrates concrete implementation details.

**Best Practices:** Use primary + secondary metrics and justify tradeoff.

**Common Mistakes:** Reporting only leaderboard rank without metric definitions.

In [3]:
import pandas as pd
from pathlib import Path

ranking_path = Path('outputs/benchmarks/model_ranking.csv')
ranking = pd.read_csv(ranking_path)
print('Top 5 models by R²:')
print(ranking[['Model','Source','R²','RMSE','MAE','MAPE']].head())

Top 5 models by R²:
           Model  Source        R²      RMSE       MAE      MAPE
0       LightGBM  Manual  0.836246  0.463233  0.308637  0.181080
1        XGBoost  Manual  0.830712  0.470996  0.313379  0.181579
2    Extra Trees  Manual  0.808230  0.501295  0.325320  0.184118
3       CatBoost  Manual  0.807484  0.502269  0.338217  0.194437
4  Random Forest  Manual  0.805505  0.504845  0.327651  0.189125
